## Monthly Order Summary
For each of the customer, produce the following summary per month
1. total orders
1. total items bought
1. total amount spent

In [0]:
df_orders = spark.table('gizmobox.silver.py_orders')
display(df_orders.filter(df_orders.customer_id == 7207))

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
15,Completed,Bank Transfer,1199,2024-10-20 01:47:25,7207,9,Smart TV,1199,1,Electronics,GoPro,White


In [0]:
print(df_orders.count())

124


In [0]:
from pyspark.sql import functions as F

x = df_orders.groupBy(['customer_id','brand']).agg(F.sum('quantity').alias('total_items_bought'))

x.display()            


customer_id,brand,total_items_bought
6627,Sony,2
6384,Sony,1
3532,Canon,2
6627,Apple,3
5028,LG,1
6167,GoPro,4
9150,HP,3
4996,Dell,1
9605,Bose,2
6384,Bose,2


In [0]:

df_order_summary = (
    df_orders
    .withColumn("order_month", F.date_format('transaction_timestamp', 'yyyy-MM'))
    .groupBy('order_month', 'customer_id')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.sum('quantity').alias('total_items_bought'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_amount')
    )
)

display(df_order_summary)

order_month,customer_id,total_orders,total_items_bought,total_amount
2024-11,9179,1,1,999
2024-12,2054,1,2,398
2024-10,5816,2,4,1296
2025-01,9084,2,4,1796
2025-01,6384,2,3,1297
2024-11,8580,1,4,586
2024-11,2639,1,6,984
2025-01,9605,2,7,1823
2025-01,9247,1,2,258
2024-11,4914,2,2,398


In [0]:
df_order_summary.writeTo('gizmobox.gold.py_order_summary_monthly').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.gold.py_order_summary_monthly;

order_month,customer_id,total_orders,total_items_bought,total_amount
2024-11,9179,1,1,999
2024-12,2054,1,2,398
2024-10,5816,2,4,1296
2025-01,9084,2,4,1796
2025-01,6384,2,3,1297
2024-11,8580,1,4,586
2024-11,2639,1,6,984
2025-01,9605,2,7,1823
2025-01,9247,1,2,258
2024-11,4914,2,2,398
